# 02 — Train DenseNet121 (5-Fold Cross-Validation)

This notebook trains DenseNet121 using a two-phase strategy:
1. **Head Training** — Base frozen, new classification head trained.
2. **Fine-Tuning** — Full model unfrozen, trained with a very low learning rate.


## 1. Vérification GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')
else:
    for gpu in gpus:
        print(f'  {gpu.name}')

## 2. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cellule unique de configuration utilisateur

In [ ]:
REPO_URL = (
    "https://github.com/MyElhadri/"
    "histology-ai-classification.git"
)

REPO_BRANCH = "main"

DRIVE_DATASET = (
    "/content/drive/MyDrive/"
    "histology-ai-classification/data/"
    "nuinsseg_human_22_original"
)

DRIVE_RESULTS = (
    "/content/drive/MyDrive/histology-results"
)

RUN_MODE = "selected_folds"

TRAIN_SINGLE_FOLD = 0

SELECTED_FOLDS = [0, 3, 4]

SEED_OVERRIDE = 42

CONFIRM_FULL_TRAINING = False

## 4. Clonage du dépôt main dans /content

In [ ]:
import os
import shutil

REPO_DIR = '/content/histology-ai-classification'
if os.path.exists(REPO_DIR):
    print(f'Removing old repo at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

print(f'Cloning {REPO_URL} (branch {REPO_BRANCH}) to {REPO_DIR}')
!git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 5. Installation des dépendances

In [ ]:
!pip install -q -r requirements-colab.txt

## 6. Copie du dataset depuis Drive vers /content

In [ ]:
import os
import shutil

DEST_DATASET = '/content/histology-ai-classification/data/raw/nuinsseg_human_22_original'

if not os.path.exists(DRIVE_DATASET):
    raise FileNotFoundError(f'Source Drive dataset missing: {DRIVE_DATASET}')

print(f'Copying dataset from {DRIVE_DATASET} to {DEST_DATASET}...')
if os.path.exists(DEST_DATASET):
    shutil.rmtree(DEST_DATASET)
shutil.copytree(DRIVE_DATASET, DEST_DATASET)
print('Copy complete.')

## 7. Vérification 22 classes et 432 images

In [ ]:
from pathlib import Path

ds_path = Path(DEST_DATASET)
folders = [f for f in ds_path.iterdir() if f.is_dir()]
if len(folders) != 22:
    raise ValueError(f'Expected 22 folders, found {len(folders)}')

image_count = sum(1 for f in ds_path.rglob('*') if f.is_file())
if image_count != 432:
    raise ValueError(f'Expected 432 images, found {image_count}')

print(f'Dataset verified: 22 folders, {image_count} images.')

## 8. Vérification du fichier des folds

In [ ]:
import pandas as pd
import os

folds_file = '/content/histology-ai-classification/data/manifests/densenet121_folds.csv'
if not os.path.exists(folds_file):
    raise FileNotFoundError(f'Folds file missing: {folds_file}')

df = pd.read_csv(folds_file)
if len(df) != 432:
    raise ValueError(f'Expected 432 rows in folds file, found {len(df)}')

expected_folds = {0, 1, 2, 3, 4}
actual_folds = set(df['fold'].unique())
if expected_folds != actual_folds:
    raise ValueError(f'Expected folds {expected_folds}, found {actual_folds}')

for idx, row in df.iterrows():
    img_path = Path(row['image_path'])
    full_path = Path('/content/histology-ai-classification') / img_path
    if not full_path.exists():
        raise FileNotFoundError(f'Missing image path in folds: {full_path}')

print('Folds file verified: 432 rows, folds 0-4 present, all image paths exist.')

## 9. Chargement du YAML

In [ ]:
from src.utils.config import load_yaml
CONFIG_PATH = 'configs/experiments/densenet121_exp_e1_regularized_crt.yaml'
config = load_yaml(CONFIG_PATH)
if SEED_OVERRIDE is not None:
    config["_seed_override"] = SEED_OVERRIDE
    actual_seed = SEED_OVERRIDE
else:
    actual_seed = config["project"]["seed"]

print("CONFIG_PATH:", CONFIG_PATH)
head_cfg = config.get("model", {}).get("classifier_head") or config.get("classifier_head", {"type": "baseline"})
ft_cfg = config.get("fine_tuning", {})
crt_cfg = config.get("classifier_retraining", {})

print(f"Experiment: {config.get('experiment_name', 'E1 regularized cRT')}")
print(f"Head: {head_cfg.get('type', 'baseline')}")
print(f"Augmentation: {config.get('augmentation', {}).get('policy', 'baseline')}")
print(f"Fine-tuning: {ft_cfg.get('strategy', 'full')}")
print(f"Label smoothing: {config.get('loss', {}).get('label_smoothing', 0.05)}")
print(f"Representation sampling: {config.get('sampling', {}).get('representation_phase', {}).get('strategy', 'natural')}")
print(f"Representation class weights: {config.get('sampling', {}).get('representation_phase', {}).get('class_weights', False)}")
print(f"cRT enabled: {crt_cfg.get('enabled', False)}")
print(f"cRT sampling: {crt_cfg.get('sampler', {}).get('strategy', 'square_root')}")
print(f"cRT class weights: {crt_cfg.get('class_weights', {}).get('enabled', False)}")
print("cRT trainable layer: predictions")
print("cRT trainable params: 2838")
print(f"Actual seed: {actual_seed}")


In [ ]:
import tensorflow as tf
from src.models.densenet121 import build_densenet121, validate_model_matches_config, apply_fine_tuning_strategy, validate_fine_tuning_strategy

head_cfg = config.get("model", {}).get("classifier_head") or config.get("classifier_head", {"type": "baseline"})
test_model = build_densenet121(
    num_classes=config["data"]["num_classes"],
    input_shape=tuple(config["data"]["image_size"]) + (3,),
    weights=None,
    head_config=head_cfg
)
validate_model_matches_config(test_model, config)

ft_cfg = config.get("fine_tuning", {})
req_ft = ft_cfg.get("strategy", "full")
apply_fine_tuning_strategy(
    test_model,
    strategy=req_ft,
    trainable_layer_prefixes=ft_cfg.get("trainable_layer_prefixes", ["conv5_"]),
    keep_batch_normalization_frozen=ft_cfg.get("keep_batch_normalization_frozen", True)
)
validate_fine_tuning_strategy(test_model, config)

tot_p = test_model.count_params()
trn_p = sum(tf.keras.backend.count_params(w) for w in test_model.trainable_weights)
assert trn_p < tot_p, f"Expected trainable_params ({trn_p}) < total_params ({tot_p}) when BatchNormalization layers are frozen."
print(f"Model pre-verification successful! Total: {tot_p}, Trainable Phase 2: {trn_p}")


## 10. Choix du mode d’exécution & Entraînement

In [ ]:
import copy
from pathlib import Path
from src.training.train import train_fold, run_cross_validation
import os

os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Run mode: {RUN_MODE}')

if RUN_MODE == 'smoke_test':
    print('Running Smoke Test (Fold 0, Batch 8, 1 epoch, class_weights enabled, separate folder)...')
    smoke_config = copy.deepcopy(config)
    smoke_config['training']['head_epochs'] = 1
    smoke_config['training']['fine_tuning_epochs'] = 0
    smoke_config['training']['batch_size'] = 8
    smoke_config['training']['use_class_weights'] = True
    out_dir = Path(DRIVE_RESULTS) / 'smoke_test'
    train_fold(smoke_config, fold=0, output_dir=out_dir)

elif RUN_MODE == 'single_fold':
    print(f'Training single fold: {TRAIN_SINGLE_FOLD} using YAML hyperparameters')
    out_dir = Path(DRIVE_RESULTS)
    train_fold(config, fold=TRAIN_SINGLE_FOLD, output_dir=out_dir)

elif RUN_MODE == 'all_folds':
    if not CONFIRM_FULL_TRAINING:
        raise ValueError('CONFIRM_FULL_TRAINING must be True to run all_folds mode.')
    print('Running full 5-fold cross-validation...')
    run_cross_validation(CONFIG_PATH, DRIVE_RESULTS)

elif RUN_MODE == 'selected_folds':
    import gc, shutil, json, subprocess
    import tensorflow as tf
    from src.models.densenet121 import build_densenet121, validate_model_matches_config, apply_fine_tuning_strategy, validate_fine_tuning_strategy, set_trainable_layers
    from src.utils.serialization import convert_numpy_to_python
    
    seed_str = f"seed{config.get('_seed_override', config['project']['seed'])}"
    folder_name = f"densenet121-exp-e1-regularized-crt-screening-{seed_str}"
    out_dir = Path(DRIVE_RESULTS) / folder_name
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy(CONFIG_PATH, out_dir / 'used_config.yaml')
    print(f'Running Selected Folds mode: {SELECTED_FOLDS} into {out_dir}')
    
    selected_metrics = {}
    pre_metrics_by_fold = {}
    post_metrics_by_fold = {}
    deltas_by_fold = {}
    
    metrics_dir = out_dir / 'reports' / 'densenet121' / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    
    for fold in SELECTED_FOLDS:
        print(f'\n--- Training Fold {fold} ---')
        metrics = train_fold(config, fold=fold, output_dir=out_dir)
        selected_metrics[f'fold_{fold}'] = metrics
        
        pre_path = metrics_dir / f"pre_crt_fold_{fold}.json"
        post_path = metrics_dir / f"post_crt_fold_{fold}.json"
        if pre_path.is_file() and post_path.is_file():
            with open(pre_path) as pf:
                pre_m = json.load(pf)
            with open(post_path) as pf:
                post_m = json.load(pf)
            pre_metrics_by_fold[f"fold_{fold}"] = pre_m
            post_metrics_by_fold[f"fold_{fold}"] = post_m
            deltas_by_fold[f"fold_{fold}"] = {
                "accuracy": post_m["accuracy"] - pre_m["accuracy"],
                "macro_f1": post_m["macro_f1"] - pre_m["macro_f1"],
                "weighted_f1": post_m["weighted_f1"] - pre_m["weighted_f1"],
                "minority_macro_f1": post_m.get("minority_macro_f1", 0) - pre_m.get("minority_macro_f1", 0),
                "minority_macro_recall": post_m.get("minority_macro_recall", 0) - pre_m.get("minority_macro_recall", 0),
                "zero_f1_class_count": post_m.get("zero_f1_class_count", 0) - pre_m.get("zero_f1_class_count", 0)
            }
            
        tf.keras.backend.clear_session()
        gc.collect()
    
    avg_acc = sum(m['accuracy'] for m in selected_metrics.values()) / len(selected_metrics)
    avg_macro = sum(m['macro_f1'] for m in selected_metrics.values()) / len(selected_metrics)
    avg_weighted = sum(m['weighted_f1'] for m in selected_metrics.values()) / len(selected_metrics)
    
    head_cfg = config.get("model", {}).get("classifier_head") or config.get("classifier_head", {"type": "baseline"})
    dummy_model = build_densenet121(
        num_classes=config["data"]["num_classes"],
        input_shape=tuple(config["data"]["image_size"]) + (3,),
        weights=None,
        dropout_rate=config["model"]["dropout_rate"],
        head_config=head_cfg
    )
    
    set_trainable_layers(dummy_model, trainable=False)
    trainable_head_params = sum(int(tf.keras.backend.count_params(w)) for w in dummy_model.trainable_weights)
    
    ft_cfg = config.get("fine_tuning", {})
    apply_fine_tuning_strategy(dummy_model, strategy=ft_cfg.get("strategy", "full"), keep_batch_normalization_frozen=ft_cfg.get("keep_batch_normalization_frozen", True))
    
    total_model_params = int(dummy_model.count_params())
    trainable_rep_params = sum(int(tf.keras.backend.count_params(w)) for w in dummy_model.trainable_weights)
    non_trainable_rep_params = sum(int(tf.keras.backend.count_params(w)) for w in dummy_model.non_trainable_weights)
    
    backbone_layers = [l for l in dummy_model.layers if not l.name.startswith(("global_average_pooling", "classifier_", "predictions"))]
    trainable_backbone_layers = [l for l in backbone_layers if l.trainable]
    trainable_backbone_params = sum(int(tf.keras.backend.count_params(w)) for l in trainable_backbone_layers for w in l.trainable_weights)
    
    backbone_bn_layers = [l for l in backbone_layers if isinstance(l, tf.keras.layers.BatchNormalization)]
    trainable_backbone_bn_layers = [l for l in backbone_bn_layers if l.trainable]
    
    for l in dummy_model.layers:
        l.trainable = (l.name == "predictions")
    trainable_crt_params = sum(int(tf.keras.backend.count_params(w)) for w in dummy_model.trainable_weights)
    
    try:
        git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode("utf-8").strip()
    except Exception:
        git_commit = "unknown"
        
    summary = {
        'experiment_name': config.get("experiment_name", "densenet121_exp_e1_regularized_crt"),
        'config_path': CONFIG_PATH,
        'requested_head_type': head_cfg.get("type", "baseline"),
        'actual_head_type': "article_inspired",
        'augmentation_policy': config.get("augmentation", {}).get("policy", "rich"),
        'requested_fine_tuning_strategy': ft_cfg.get("strategy", "full"),
        'actual_fine_tuning_strategy': ft_cfg.get("strategy", "full"),
        'seed': config.get("_seed_override", config["project"]["seed"]),
        'folds_executed': SELECTED_FOLDS,
        'total_params': total_model_params,
        'trainable_params_head_phase': trainable_head_params,
        'trainable_params_representation_phase': trainable_rep_params,
        'non_trainable_params_representation_phase': non_trainable_rep_params,
        'trainable_backbone_params_representation_phase': trainable_backbone_params,
        'trainable_backbone_bn_count_representation_phase': len(trainable_backbone_bn_layers),
        'trainable_params_crt_phase': trainable_crt_params,
        'label_smoothing': config.get("loss", {}).get("label_smoothing", 0.05),
        'representation_sampling_strategy': config.get("sampling", {}).get("representation_phase", {}).get("strategy", "natural"),
        'representation_class_weights_enabled': config.get("sampling", {}).get("representation_phase", {}).get("class_weights", False),
        'crt_enabled': config.get("classifier_retraining", {}).get("enabled", True),
        'crt_sampling_strategy': config.get("classifier_retraining", {}).get("sampler", {}).get("strategy", "square_root"),
        'crt_frequency_power': config.get("classifier_retraining", {}).get("sampler", {}).get("frequency_power", -0.5),
        'crt_class_weights_enabled': config.get("classifier_retraining", {}).get("class_weights", {}).get("enabled", False),
        'crt_predictions_reinitialized': config.get("classifier_retraining", {}).get("reinitialize_predictions", True),
        'checkpoint_monitor': "val_ce_hard",
        'pre_crt_metrics': pre_metrics_by_fold,
        'post_crt_metrics': post_metrics_by_fold,
        'crt_delta': deltas_by_fold,
        'metrics': selected_metrics,
        'average': {
            'accuracy': float(avg_acc),
            'macro_f1': float(avg_macro),
            'weighted_f1': float(avg_weighted)
        },
        'git_commit': git_commit,
        'tf_version': tf.__version__
    }
    with open(metrics_dir / 'selected_folds_summary.json', 'w') as f:
        json.dump(convert_numpy_to_python(summary), f, indent=2)

else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

## 11. Affichage des résultats

In [ ]:
import json
from pathlib import Path

metrics_dir = Path(DRIVE_RESULTS) / 'reports' / 'densenet121' / 'metrics'

if not metrics_dir.exists():
    print(f'Metrics directory not found: {metrics_dir}')
else:
    summary_path = metrics_dir / 'cross_validation_summary.json'
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print('Average Accuracy:', summary['average']['accuracy'])
        print('Average Macro F1:', summary['average']['macro_f1'])
    else:
        print('Summary not available (run mode was not all_folds or training incomplete).')
        for fold_file in sorted(metrics_dir.glob('fold_*.json')):
            if 'class_weights' in fold_file.name:
                continue
            with open(fold_file) as f:
                m = json.load(f)
            print(f'{fold_file.stem}: accuracy={m.get("accuracy", "N/A")}, macro_f1={m.get("macro_f1", "N/A")}')
